# 15 - OpenAI Agents SDK

| | |
|---|---|
| **Pattern** | vendor SDK: agents with **handoffs**, **agents-as-tools** and **guardrails** as first-class concepts |
| **Communication** | handoff = control transfer (like 04); `as_tool` = encapsulated call (like 06); guardrails run *beside* the agent |
| **Use when** | you want OpenAI's opinionated, very small API - it runs fine on any Chat Completions endpoint |
| **Watch out** | **tracing is on by default and uploads your runs to OpenAI**, even when the model is Foundry or Ollama |

Three vendors, three harnesses - samples 15-17 are the companies' own agent
SDKs rather than neutral frameworks:

| | OpenAI (this) | Anthropic (16) | GitHub (17) |
|---|---|---|---|
| shape | small library: you define agents and tools | Claude Code as a library: built-in file/shell tools | Copilot CLI as a library |
| model | any Chat Completions / Responses endpoint | Claude | Copilot's models |
| runs offline here | **yes** (mock) | no - needs Claude auth | no - needs Copilot auth |

```bash
uv run 15_openai_agents_sdk.py --profile mock
```

In [ ]:
import asyncio

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OpenAIChatCompletionsModel,
    Runner,
    function_tool,
    input_guardrail,
    set_tracing_disabled,
)
from openai import AsyncOpenAI

from genaiclass import banner, get_profile

profile = get_profile()
print(banner(profile))

## Pointing it away from OpenAI - and switching off the upload

`OpenAIChatCompletionsModel` takes any `AsyncOpenAI` client, so the course
endpoint, Ollama or the mock work unchanged. What does **not** change by
itself: the SDK's tracing exporter sends every run (prompts, tool calls,
outputs) to OpenAI's trace dashboard using `OPENAI_API_KEY`. With a non-OpenAI
model that is at best 401 noise and at worst your data leaving the building.
Switch it off explicitly - or plug in your own exporter (OpenTelemetry, Langfuse).

In [ ]:
set_tracing_disabled(True)

model = OpenAIChatCompletionsModel(
    model=profile.model,
    openai_client=AsyncOpenAI(base_url=profile.base_url, api_key=profile.api_key,
                              default_headers=profile.headers or None),
)

## Tools and agents

`@function_tool` builds the schema from the signature and docstring, as MAF's
`@tool` does. `handoff_description` is what *other* agents see when deciding
whether to hand off to this one.

In [ ]:
@function_tool
def get_current_weather(location: str) -> str:
    """Current weather conditions for one place."""
    return f"18C and light rain in {location}"


weather = Agent(name="weather", model=model, tools=[get_current_weather],
                instructions="You answer weather questions using the tool. Be brief.",
                handoff_description="Weather: forecasts, rain, temperature")
billing = Agent(name="billing", model=model,
                instructions="You resolve invoice and payment questions. Be brief.",
                handoff_description="Billing: invoices, payments, refunds, charges")

## A guardrail runs beside the agent, not inside it

An input guardrail sees the input before (or in parallel with) the agent and
can trip a wire that aborts the run with an exception. It can be a model call
(a cheap classifier agent) or plain code - here, plain code, so it is free and
deterministic.

In [ ]:
@input_guardrail
async def no_card_numbers(ctx, agent, user_input) -> GuardrailFunctionOutput:
    text = user_input if isinstance(user_input, str) else str(user_input)
    digits = sum(ch.isdigit() for ch in text)
    return GuardrailFunctionOutput(output_info={"digits": digits}, tripwire_triggered=digits >= 12)


triage = Agent(
    name="triage",
    model=model,
    instructions="You are the front desk. Hand off to the right specialist.",
    handoffs=[weather, billing],           # control transfer, as in sample 04
    input_guardrails=[no_card_numbers],
)

# The same specialists, used the other way round: as tools of a supervisor (sample 06).
supervisor = Agent(
    name="supervisor",
    model=model,
    instructions="Answer using your tools, then combine the results in one reply.",
    tools=[weather.as_tool(tool_name="ask_weather", tool_description="Ask the weather specialist."),
           billing.as_tool(tool_name="ask_billing", tool_description="Ask the billing specialist about invoices.")],
)

## Run all three behaviours

`result.last_agent` shows who ended up answering - after a handoff that is
the specialist, not triage. `to_input_list()` is the conversation so far; pass
it back in for the next turn (the SDK's built-in sessions now store history
on OpenAI's side, which would not work against other providers).

In [ ]:
async def main() -> None:
    print("--- 1. handoff: control moves to the specialist")
    result = await Runner.run(triage, "My invoice looks wrong.")
    print(f"  answered by: {result.last_agent.name}\n  {result.final_output}")

    follow_up = result.to_input_list() + [{"role": "user", "content": "It shows the charge twice."}]
    result = await Runner.run(result.last_agent, follow_up)
    print(f"  turn 2 stays with: {result.last_agent.name} ({len(follow_up)} items sent)")

    print("\n--- 2. agents as tools: the supervisor keeps control")
    result = await Runner.run(supervisor, "What is the weather in Lisbon, and why is my invoice charged twice?")
    print(f"  answered by: {result.last_agent.name}\n  {str(result.final_output)[:160]}")

    print("\n--- 3. guardrail: the run never reaches a model")
    try:
        await Runner.run(triage, "Refund card 4111 1111 1111 1111 please")
    except InputGuardrailTripwireTriggered as tripped:
        print(f"  blocked by guardrail: {tripped.guardrail_result.output.output_info}")


asyncio.run(main())

## What the SDK adds beyond this sample

* **Sandboxed harness (2026)**: Codex-style file and shell tools with sandbox
  providers (E2B, Modal, Daytona, ...), comparable to what sample 16 gets from
  Claude Code - Python only at the time of writing.
* **Tracing**: excellent when you *are* on OpenAI; replace the exporter otherwise.
* **Hosted twin**: the same agents run as Foundry hosted agents (sample 12's
  container accepts OpenAI Agents SDK code too).